# Chapter 06 제출 답안 — 데이터를 보며 질문을 만드는 EDA

- 이름: 송누리
- GitHub ID: HauserNR
- 작성일: 2026-10-21
- 최종 Notebook URL: https://github.com/HauserNR/llm-data-analysis-study/blob/main/chapter06/chapter06.ipynb

Chapter 01~05의 개인 저장소를 이어 사용한다. 특히 Chapter 05의 전용 Raw로 만든 processed CSV 4개를 루트 `data/processed/`에 바이트 그대로 복사했다. 입력 해시와 PK/FK, 날짜 실패, 극단값을 다시 확인한다. 공통 Raw로 새로 전처리하지 않았다.

공식 설명·과제 문구와 원본 58셀의 순서를 유지했다. 경로 설정, completed 필터, 결측 그룹 보존, 병합 검증, 모듈 경로, 과제 풀이 코드는 과제 조건에 맞게 보완하고 `official_cell_index`로 추적한다. 추가 답안과 검증은 관련 결과 뒤에 둔다. 이 Notebook의 금액은 **전처리 후 남은 completed 주문 상세 금액**이며 회계상 확정 매출이나 원본 전체 금액을 뜻하지 않는다.

공식 자료: [가이드](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter06/chapter06.md), [Notebook](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/notebooks/ch06_eda_questions.ipynb), [답안 템플릿](https://github.com/GilbertMoon/llm-data-analysis-course/blob/main/practice/chapter06/templates/chapter06_assignment.md). 내려받은 원본은 `references/`에 보관한다. 코드 보완·질문 후보·답안 초안에 Codex를 사용했다. 결과 수치는 아래 실제 실행 출력으로 검증한다.


# 6장. 데이터를 보며 질문을 만드는 EDA

이 노트북은 `book/chapters/ch06_eda_questions.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 그래프나 표를 많이 만드는 것이 아니라, 데이터를 보며 **현재 데이터로 답할 수 있는 질문**을 만들고 그 질문을 지표와 코드로 연결하는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 이 노트북은 5장에서 만든 `data/processed/*_clean.csv` 파일을 사용합니다.
- 전처리 파일이 없다면 먼저 `python scripts/preprocess_data.py`를 실행하세요.
- EDA 결과표는 `reports/` 폴더에 CSV와 Markdown 보고서로 저장합니다.
- EDA 결과는 최종 결론이 아니라, 다음 분석 질문을 만드는 중간 산출물입니다.


## 1. EDA는 결론보다 질문에 가깝다

EDA는 Exploratory Data Analysis, 즉 탐색적 데이터 분석입니다. 이름 그대로 데이터를 여러 방향에서 탐색하면서 분포, 패턴, 차이, 관계, 이상한 값을 발견하는 과정입니다.

EDA에서 자주 던지는 질문은 다음과 같습니다.

- 데이터는 어떤 변수들로 구성되어 있는가?
- 주요 숫자형 변수의 분포는 어떤가?
- 주요 범주형 변수의 빈도는 어떤가?
- 특정 그룹 간 차이가 있는가?
- 시간에 따른 변화가 있는가?
- 이상하게 큰 값이나 작은 값이 있는가?
- 다음 단계에서 더 깊이 볼 질문은 무엇인가?


## 2. 관찰, 가설, 결론 구분하기

EDA 결과를 해석할 때는 관찰, 가설, 결론을 구분해야 합니다.

| 구분 | 의미 | 예시 |
|---|---|---|
| 관찰 | 데이터에서 직접 확인한 사실 | 전자기기 카테고리의 매출 비중이 가장 높다 |
| 가설 | 관찰을 바탕으로 생각해 볼 가능성 | 전자기기는 단가가 높아 매출 비중이 클 수 있다 |
| 결론 | 추가 검증 후 말할 수 있는 판단 | 단가와 판매 수량을 함께 분석한 결과 매출 차이의 주요 요인은 단가였다 |

이번 장에서는 주로 관찰과 가설을 만들고, 다음 분석 질문으로 연결하는 연습을 합니다.


## 3. 좋은 분석 질문의 조건

좋은 분석 질문은 데이터로 답할 수 있어야 합니다.

| 조건 | 설명 | 예시 |
|---|---|---|
| 데이터 기반 | 현재 데이터로 답할 수 있어야 함 | 월별 매출은 어떻게 변하는가? |
| 구체적 | 분석 대상과 기준이 명확해야 함 | 카테고리별 매출 비중은 어떻게 다른가? |
| 측정 가능 | 지표로 계산할 수 있어야 함 | 주문 수, 총매출, 평균 구매 금액 |
| 해석 가능 | 결과가 다음 판단과 연결되어야 함 | 어떤 상품군을 더 자세히 봐야 하는가? |
| 검증 가능 | 코드로 확인할 수 있어야 함 | `groupby()`로 계산 가능 |


## 4. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
from IPython.display import display
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = next(p for p in [CURRENT_DIR, *CURRENT_DIR.parents]
                    if (p / 'chapter05/data/processed/orders_clean.csv').exists())
CHAPTER_DIR = PROJECT_ROOT / 'chapter06'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
REPORT_DIR = CHAPTER_DIR / 'reports'
REPORT_DIR.mkdir(exist_ok=True)
# chapter06/src를 우선해 이전 장의 src와 분리한다.
sys.path.insert(0, str(CHAPTER_DIR))
print('현재 위치:', CURRENT_DIR)
print('입력:', PROCESSED_DIR)
print('출력:', REPORT_DIR)

현재 위치: /Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06
입력: /Users/kjj/MyLLMGit/llm-data-analysis-study/data/processed
출력: /Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports


## 5. 전처리된 데이터 불러오기

6장은 5장에서 저장한 전처리 데이터를 사용합니다. 전처리 파일이 없다면 먼저 아래 명령을 터미널에서 실행하세요.

```bash
python scripts/preprocess_data.py
```


In [2]:
required_files = [
    PROCESSED_DIR / 'customers_clean.csv',
    PROCESSED_DIR / 'products_clean.csv',
    PROCESSED_DIR / 'orders_clean.csv',
    PROCESSED_DIR / 'order_items_clean.csv',
]

missing_files = [path for path in required_files if not path.exists()]

if missing_files:
    for path in missing_files:
        print('누락 파일:', path)
    raise FileNotFoundError('전처리 파일이 없습니다. 먼저 python scripts/preprocess_data.py 를 실행하세요.')

customers = pd.read_csv(PROCESSED_DIR / 'customers_clean.csv')
products = pd.read_csv(PROCESSED_DIR / 'products_clean.csv')
orders = pd.read_csv(PROCESSED_DIR / 'orders_clean.csv')
order_items = pd.read_csv(PROCESSED_DIR / 'order_items_clean.csv')

print('전처리 데이터 불러오기 완료')


전처리 데이터 불러오기 완료


### STEP 1. Chapter 05 입력 계승과 품질 재확인

기존 전처리에서는 고객 나이 6개를 중앙값 41로 대체하고, 날짜 실패·FK 불일치·극단값은 임의 복구하지 않았다. 아래 검사로 입력의 동일성을 확인한다. `clean`이라는 파일명만으로 분석 적합성을 판단하지 않는다.

In [3]:
import hashlib
input_manifest = []
for name, frame in [('customers',customers),('products',products),('orders',orders),('order_items',order_items)]:
    filename = name + '_clean.csv'
    old = PROJECT_ROOT / 'chapter05/data/processed' / filename
    new = PROCESSED_DIR / filename
    assert old.read_bytes() == new.read_bytes()
    pk = {'customers':'customer_id','products':'product_id','orders':'order_id','order_items':'order_item_id'}[name]
    assert frame[pk].notna().all() and frame[pk].is_unique
    input_manifest.append({'file':filename,'rows':len(frame),'sha256':hashlib.sha256(new.read_bytes()).hexdigest()})
assert np.allclose(order_items.line_total, order_items.quantity * order_items.unit_price)
pd.DataFrame(input_manifest).to_csv(REPORT_DIR/'input_manifest.csv',index=False)
display(pd.DataFrame(input_manifest)[['file','rows']])
print('FK 불일치 (주문→고객 / 상세→주문 / 상세→상품):',
      (~orders.customer_id.isin(customers.customer_id)).sum(),
      (~order_items.order_id.isin(orders.order_id)).sum(),
      (~order_items.product_id.isin(products.product_id)).sum())
print('입력 4개 바이트 일치 / PK 유일성 / line_total 산식 검증 통과')

,file,rows
0,customers_clean.csv,150
1,products_clean.csv,96
2,orders_clean.csv,300
3,order_items_clean.csv,752


FK 불일치 (주문→고객 / 상세→주문 / 상세→상품): 2 2 26
입력 4개 바이트 일치 / PK 유일성 / line_total 산식 검증 통과


### 결과 관찰
고객 150행, 상품 96행, 주문 300행, 상세 752행이며 Chapter 05 파일과 바이트가 일치한다. FK 불일치가 각각 2·2·26건 남는다.

### 나의 해석과 판단
Chapter 05에서 결측·비양수 행을 제외한 결과를 계승한다. 남은 FK 오류는 키를 임의 생성해 메우지 않고 금액 집계에서 누락 여부를 측정한다.

### 업무·분석적 의미
전처리 버전이 바뀌어 결과가 달라지는 혼선을 방지한다.

### 한계와 추가 확인 사항
이미 전처리에서 제외된 행의 금액을 복원할 수 없으므로 이번 금액은 원본 전체 거래를 대표하지 않는다.

## 6. 데이터 크기와 컬럼 확인하기

EDA를 시작하기 전에 각 데이터의 행/열 수와 컬럼명을 다시 확인합니다. CSV로 저장했다가 다시 불러오면 날짜 컬럼이 문자열로 돌아올 수 있으므로 날짜형 변환도 다시 확인합니다.


In [4]:
data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

for name, df in data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (150, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (96, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (300, 7)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status', 'order_month', 'order_dayofweek']

[order_items]
shape: (752, 6)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price', 'line_total']



In [5]:
orders['order_date'] = pd.to_datetime(orders['order_date'], errors='coerce')

if 'signup_date' in customers.columns:
    customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce')

if 'order_month' not in orders.columns:
    orders['order_month'] = orders['order_date'].dt.to_period('M').astype(str).fillna('NaT')

print('order_date 변환 실패:', orders['order_date'].isna().sum())
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


## 7. 분석 질문을 먼저 표로 정리하기

EDA를 시작하기 전에 어떤 질문을 볼지 먼저 표로 정리하면 분석이 산만해지지 않습니다. 아래 표는 고객, 상품, 주문, 매출, 시간, 고객 가치 관점에서 만들 수 있는 기본 질문입니다.


In [6]:
questions = pd.DataFrame({
    'analysis_area': ['고객', '상품', '주문', '매출', '시간', '고객 가치'],
    'question': [
        '고객은 어떤 지역과 연령대에 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '주문 상태와 결제수단 분포는 어떤가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'metric': [
        '고객 수, 평균 나이',
        '카테고리별 상품 수',
        '주문 수, 비율',
        '총매출, 매출 비중',
        '월별 매출, 월별 주문 수',
        '고객별 총매출, 주문 횟수',
    ],
    'required_data': [
        'customers',
        'products',
        'orders',
        'order_items, products',
        'orders, order_items',
        'customers, orders, order_items',
    ],
})

questions


,analysis_area,question,metric,required_data
0,고객,고객은 어떤 지역과 연령대에 분포하는가?,"고객 수, 평균 나이",customers
1,상품,어떤 카테고리의 상품이 많은가?,카테고리별 상품 수,products
2,주문,주문 상태와 결제수단 분포는 어떤가?,"주문 수, 비율",orders
3,매출,카테고리별 매출은 어떻게 다른가?,"총매출, 매출 비중","order_items, products"
4,시간,월별 매출과 주문 수는 어떻게 변하는가?,"월별 매출, 월별 주문 수","orders, order_items"
5,고객 가치,구매 금액이 높은 고객은 누구인가?,"고객별 총매출, 주문 횟수","customers, orders, order_items"


## 답안 1. 내가 정한 EDA 질문

| 질문 | 분석 범위·필요 데이터 | 지표·계산 방법 | 검증 방법 |
|---|---|---|---|
| 카테고리 금액 차이는 수량과 단가 중 무엇과 함께 나타나는가? | completed 상세 + products | 수량 합, line_total 합, 금액/수량 가중 거래단가 | many_to_one 병합, Unknown 포함 합계 일치 |
| 월별 금액 차이는 주문 수와 객단가에서 어떻게 나타나는가? | completed 상세 + orders | 월별 금액, order_id nunique, 금액/주문 수 | NaT 포함 총합 대조, 미래·미상 날짜 분리 |
| 상품 수가 많은 카테고리는 금액도 높은가? | 전체 clean 상품 + completed 상세 | 상품 수와 금액 순위, Spearman 상관 | 미연결 상품 별도, 7개 그룹의 탐색적 관계로 한정 |
| 결제수단별 객단가는 어떻게 다른가? | completed 상세 + orders | 주문별 합산 후 결제수단별 mean | 금액 합계와 고유 주문 수 대조 |
| 고객이 많은 도시의 구매 금액도 높은가? | customers + completed 상세 + orders | 고객 수, 구매자 수, 총금액, 고객당·구매자당 금액 | 고객 미연결 Unknown, 도시 합계 대조 |

### 왜 이 질문을 선택했는가?
세 데이터의 키와 상태·날짜·수량·거래 단가가 있어 계산할 수 있다. 상품 가격 평균과 실제 거래 가중단가를 구분하고, 규모 차이를 고객 수·주문 수로 나누어 확인하고 싶었다. 광고·계절성·만족도는 관측 변수가 없으므로 원인 질문으로 단정하지 않는다.

**분석 범위:** 주 집계는 clean 상세 중 orders에서 completed가 확인되는 모든 행이다. 날짜 미상·미래와 상품 미연결은 버리지 않고 각각 NaT·Unknown으로 표시한다. 비교용 정상 시점 범위는 `order_date <= 2026-10-21`, 날짜 유효인 행이며 따로 총합을 제시한다. 전체 고객·상품·주문 분포는 모집단 설명으로만 사용한다.

## 8. 고객 데이터 EDA

먼저 고객 데이터만 따로 살펴봅니다. 하나의 데이터셋 안에서 분포와 빈도를 확인하면 이후 병합 분석을 할 때 기준이 생깁니다.


In [7]:
customers['age'].describe()


count    150.000000
mean      43.913333
std       17.802397
min       18.000000
25%       30.000000
50%       41.000000
75%       56.750000
max      150.000000
Name: age, dtype: float64

In [8]:
customer_city = customers['city'].value_counts(dropna=False).reset_index()
customer_city.columns = ['city', 'customer_count']
customer_city['customer_ratio'] = (
    customer_city['customer_count'] / customer_city['customer_count'].sum() * 100
).round(2)

customer_city


,city,customer_count,customer_ratio
0,부산,23,15.33
1,대구,21,14.00
2,광주,21,14.00
3,서울,18,12.00
4,울산,17,11.33
5,인천,16,10.67
6,수원,10,6.67
7,성남,9,6.00
8,고양,8,5.33
9,대전,5,3.33


In [9]:
customer_gender = customers['gender'].value_counts(dropna=False).reset_index()
customer_gender.columns = ['gender', 'customer_count']
customer_gender['customer_ratio'] = (
    customer_gender['customer_count'] / customer_gender['customer_count'].sum() * 100
).round(2)

customer_gender


,gender,customer_count,customer_ratio
0,M,80,53.33
1,F,70,46.67


고객 데이터에서 이어질 수 있는 질문은 다음과 같습니다.

- 고객은 어느 도시에 많이 분포하는가?
- 고객의 평균 나이는 어느 정도인가?
- 성별 고객 수는 어떻게 분포하는가?
- 도시별 구매 금액도 차이가 있는가?


## 9. 상품 데이터 EDA

상품 데이터에서는 카테고리와 가격을 먼저 봅니다. 상품 수가 많은 카테고리가 매출도 높은지는 아직 알 수 없습니다. 그것은 주문 상세와 연결한 뒤 확인해야 합니다.


In [10]:
product_category = products['category'].value_counts(dropna=False).reset_index()
product_category.columns = ['category', 'product_count']
product_category['product_ratio'] = (
    product_category['product_count'] / product_category['product_count'].sum() * 100
).round(2)

product_category


,category,product_count,product_ratio
0,뷰티,17,17.71
1,패션,16,16.67
2,식품,16,16.67
3,전자기기,14,14.58
4,도서,13,13.54
5,스포츠,10,10.42
6,생활용품,10,10.42


In [11]:
products['price'].describe()


count    9.600000e+01
mean     2.048958e+05
std      1.011848e+06
min      5.000000e+03
25%      5.275000e+04
50%      9.450000e+04
75%      1.532500e+05
max      9.999999e+06
Name: price, dtype: float64

In [12]:
category_price = (
    products
    .groupby('category', as_index=False)
    .agg(
        product_count=('product_id', 'count'),
        avg_price=('price', 'mean'),
        min_price=('price', 'min'),
        max_price=('price', 'max'),
    )
    .assign(avg_price=lambda df: df['avg_price'].round(0))
    .sort_values('avg_price', ascending=False)
)

category_price


,category,product_count,avg_price,min_price,max_price
1,뷰티,17,672294.0,18000.0,9999999.0
4,식품,16,120812.0,11000.0,195000.0
6,패션,16,115125.0,28000.0,197000.0
5,전자기기,14,113143.0,21000.0,178000.0
0,도서,13,103692.0,34000.0,182000.0
3,스포츠,10,78400.0,5000.0,183000.0
2,생활용품,10,75000.0,29000.0,147000.0


## 10. 주문 데이터 EDA

주문 데이터에서는 주문 상태, 결제수단, 주문 기간을 확인합니다. 주문 상태 분포는 이후 매출 분석에서 완료 주문만 볼지, 전체 주문을 볼지 결정하는 기준이 됩니다.


In [13]:
order_status = orders['order_status'].value_counts(dropna=False).reset_index()
order_status.columns = ['order_status', 'order_count']
order_status['order_ratio'] = (
    order_status['order_count'] / order_status['order_count'].sum() * 100
).round(2)

order_status


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


In [14]:
payment_method = orders['payment_method'].value_counts(dropna=False).reset_index()
payment_method.columns = ['payment_method', 'order_count']
payment_method['order_ratio'] = (
    payment_method['order_count'] / payment_method['order_count'].sum() * 100
).round(2)

payment_method


,payment_method,order_count,order_ratio
0,kakao_pay,90,30.00
1,card,72,24.00
2,bank_transfer,70,23.33
3,naver_pay,68,22.67


In [15]:
print('주문 시작일:', orders['order_date'].min())
print('주문 종료일:', orders['order_date'].max())
print('주문 월 개수:', orders['order_month'].nunique())


주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00
주문 월 개수: 16


In [16]:
# Evidence 01 — 기본 분포와 Chapter 05 잔여 품질 이슈
print('Evidence 01: 기본 분포·품질')
display(order_status)
display(customers[['age']].describe().round(2))
print('나이 120 초과:', customers.age.gt(120).sum())
print('상품 가격 100만원 초과:', products.price.gt(1_000_000).sum())
print('수량 20 초과:', order_items.quantity.gt(20).sum())
print('날짜 변환 실패:', orders.order_date.isna().sum())
print('기준일 이후 주문:', orders.order_date.gt(pd.Timestamp('2026-10-21')).sum())

Evidence 01: 기본 분포·품질


,order_status,order_count,order_ratio
0,completed,180,60.0
1,cancelled,63,21.0
2,refunded,57,19.0


,age
count,150.00
mean,43.91
std,17.80
min,18.00
25%,30.00
50%,41.00
75%,56.75
max,150.00


나이 120 초과: 1
상품 가격 100만원 초과: 1
수량 20 초과: 2
날짜 변환 실패: 3
기준일 이후 주문: 2


### 실제 VS Code 화면 — Evidence 01

주문 상태·나이 분포와 잔여 품질 이슈. 실제 실행된 Notebook을 VS Code에서 열어 코드·실행 번호·출력을 캡처했다. 원본 JPEG를 그대로 저장했으며 합성·재그리기·이미지 편집을 하지 않았다.

![Evidence 01: 주문 상태·나이 분포와 잔여 품질 이슈](images/step01_distribution.jpg)

### 결과 관찰
전체 주문 중 completed는 180건(60%), cancelled 63건, refunded 57건이다. 나이는 평균 43.91세·중앙값 41세·최댓값 150세다. 날짜 실패 3건, 기준일 이후 2건, 수량 20 초과 2건을 확인했다.

### 나의 해석과 판단
150세를 실제 고객 특성으로 단정하지 않는다. 나이 6개가 중앙값으로 대체되어 분포가 일부 바뀌었다. completed 분석에 취소·환불 금액을 섞지 않는다.

### 업무·분석적 의미
전체 주문 분포와 금액 분석 모집단을 분리해야 담당자 간 지표 해석이 일치한다.

### 한계와 추가 확인 사항
극단값의 오류 여부는 원천 주문 확인이 필요하다. 분포 요약은 원인이나 인구 대표성을 보장하지 않는다.

## 11. 매출 분석을 위한 데이터 연결

카테고리별 매출을 계산하려면 주문 상세와 상품 정보를 `product_id` 기준으로 연결해야 합니다. 병합 후에는 행 수와 누락 여부를 반드시 확인합니다.


In [17]:
if 'line_total' not in order_items.columns:
    order_items['line_total'] = order_items['quantity'] * order_items['unit_price']

order_items[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [18]:
# completed 주문 상세로 범위를 통일한다. 결측 카테고리는 Unknown으로 보존한다.
completed_items = order_items.loc[order_items['order_id'].isin(
    orders.loc[orders['order_status'].eq('completed'), 'order_id'])].copy()
sales_items = completed_items.merge(products, on='product_id', how='left', validate='many_to_one', indicator=True)
assert len(sales_items) == len(completed_items)
print('전체 상세:', len(order_items), 'completed 상세:', len(completed_items))
print('상품 미연결:', sales_items['_merge'].eq('left_only').sum())
sales_items['category'] = sales_items['category'].fillna('Unknown')
sales_items.head()

전체 상세: 752 completed 상세: 446
상품 미연결: 16


,order_item_id,order_id,product_id,quantity,unit_price,line_total,product_name,category,price,_merge
0,1,1,39,2.0,172000.0,344000.0,뷰티 상품 039,뷰티,172000.0,both
1,2,2,26,4.0,79000.0,316000.0,뷰티 상품 026,뷰티,79000.0,both
2,3,2,46,3.0,87000.0,261000.0,생활용품 상품 046,생활용품,87000.0,both
3,4,2,80,1.0,187000.0,187000.0,패션 상품 080,패션,187000.0,both
4,5,2,91,5.0,125000.0,625000.0,전자기기 상품 091,전자기기,125000.0,both


## 12. 카테고리별 매출 EDA

카테고리별 매출은 어떤 상품군이 매출에 많이 기여했는지 보여 줍니다. 다만 매출이 높은 이유가 판매 수량 때문인지, 단가 때문인지는 추가로 확인해야 합니다.


In [19]:
category_sales = (
    sales_items
    .groupby('category', as_index=False)
    .agg(
        total_quantity=('quantity', 'sum'),
        total_sales=('line_total', 'sum'),
    )
    .sort_values('total_sales', ascending=False)
)

category_sales['sales_ratio'] = (
    category_sales['total_sales'] / category_sales['total_sales'].sum() * 100
).round(2)

category_sales


,category,total_quantity,total_sales,sales_ratio
7,패션,290.0,32940000.0,23.31
5,식품,226.0,25035000.0,17.71
6,전자기기,181.0,21590000.0,15.28
2,뷰티,180.0,17751000.0,12.56
1,도서,174.0,17663000.0,12.50
4,스포츠,165.0,13237000.0,9.37
3,생활용품,125.0,9109000.0,6.44
0,Unknown,34.0,4015000.0,2.84


In [20]:
category_sales_with_price = category_sales.merge(
    category_price[['category', 'avg_price']],
    on='category',
    how='left',
)

category_sales_with_price


,category,total_quantity,total_sales,sales_ratio,avg_price
0,패션,290.0,32940000.0,23.31,115125.0
1,식품,226.0,25035000.0,17.71,120812.0
2,전자기기,181.0,21590000.0,15.28,113143.0
3,뷰티,180.0,17751000.0,12.56,672294.0
4,도서,174.0,17663000.0,12.50,103692.0
5,스포츠,165.0,13237000.0,9.37,78400.0
6,생활용품,125.0,9109000.0,6.44,75000.0
7,Unknown,34.0,4015000.0,2.84,NaN


In [21]:
# Evidence 02 — completed 카테고리 수량·금액·실제 가중 거래단가
print('Evidence 02: completed 카테고리 집계')
category_detail = category_sales.assign(weighted_unit_price=lambda x:x.total_sales/x.total_quantity)
display(category_detail.round(2))
print('completed 원본 합계:', f'{completed_items.line_total.sum():,.0f}')

Evidence 02: completed 카테고리 집계


,category,total_quantity,total_sales,sales_ratio,weighted_unit_price
7,패션,290.0,32940000.0,23.31,113586.21
5,식품,226.0,25035000.0,17.71,110774.34
6,전자기기,181.0,21590000.0,15.28,119281.77
2,뷰티,180.0,17751000.0,12.56,98616.67
1,도서,174.0,17663000.0,12.50,101511.49
4,스포츠,165.0,13237000.0,9.37,80224.24
3,생활용품,125.0,9109000.0,6.44,72872.00
0,Unknown,34.0,4015000.0,2.84,118088.24


completed 원본 합계: 141,340,000


### 실제 VS Code 화면 — Evidence 02

completed 카테고리 수량·금액·가중 거래단가. 실제 실행된 Notebook을 VS Code에서 열어 코드·실행 번호·출력을 캡처했다. 원본 JPEG를 그대로 저장했으며 합성·재그리기·이미지 편집을 하지 않았다.

![Evidence 02: completed 카테고리 수량·금액·가중 거래단가](images/step02_category.jpg)

## 답안 2. 핵심 집계 결과 — 카테고리

### 결과 관찰
패션 32,940,000원(23.31%)·290개, 식품 25,035,000원(17.71%)·226개, 전자기기 21,590,000원(15.28%)·181개 순이다. 패션의 가중 거래단가는 113,586.21원, 전자기기는 119,281.77원이다. 상품 미연결 completed 상세 16행의 4,015,000원(2.84%)은 Unknown으로 남겼다.

### 나의 해석과 판단
패션이 거래단가까지 가장 높은 것은 아니다. 현재 표본에서 패션의 금액 1위는 수량 1위와 함께 나타난다. 상품 마스터 평균 가격은 미판매 상품·가격 극단값의 영향을 받으므로 거래 단가의 대용으로 사용하지 않는다.

### 업무·분석적 의미
금액 1위라는 사실만으로 가격 인상·재고 확대를 결정하기보다 수량과 개별 상품 기여도를 확인할 가치가 있다.

### 한계와 추가 확인 사항
가설은 선호도나 인과 설명이 아니다. 상품 미연결 2.84%가 특정 카테고리에 몰렸을 수 있다. 수량 극단값 50개가 패션 금액에 미치는 영향은 아래 과제 5에서 검증한다.

해석 예시:

> 관찰: 매출 비중이 높은 카테고리가 존재합니다.
> 가설: 해당 카테고리는 판매 수량이 많거나 평균 단가가 높아서 매출이 클 수 있습니다.
> 추가 분석: 카테고리별 판매 수량과 평균 단가를 함께 비교해야 합니다.


## 13. 월별 매출과 주문 수 EDA

월별 매출을 보려면 주문 상세와 주문 정보를 `order_id` 기준으로 연결합니다. 월별 매출은 시간 흐름을 보여 주지만, 원인을 바로 설명하지는 않습니다.


In [22]:
order_sales = completed_items.merge(orders, on='order_id', how='left', validate='many_to_one')
assert len(order_sales) == len(completed_items)
assert order_sales['order_status'].eq('completed').all()
order_sales['order_date'] = pd.to_datetime(order_sales['order_date'], errors='coerce')
# NaT는 명시적 그룹으로 남겨 completed 원본 합계와 같은 범위를 유지한다.
order_sales['order_month'] = order_sales['order_date'].dt.to_period('M').astype(str).fillna('NaT')
print('completed 주문 상세:', order_sales.shape)
print('날짜 불명 상세:', order_sales['order_date'].isna().sum())
order_sales.head()

completed 주문 상세: (446, 12)
날짜 불명 상세: 6


,order_item_id,order_id,product_id,quantity,unit_price,line_total,customer_id,order_date,payment_method,order_status,order_month,order_dayofweek
0,1,1,39,2.0,172000.0,344000.0,144,2026-07-12,naver_pay,completed,2026-07,Sunday
1,2,2,26,4.0,79000.0,316000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday
2,3,2,46,3.0,87000.0,261000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday
3,4,2,80,1.0,187000.0,187000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday
4,5,2,91,5.0,125000.0,625000.0,149,2025-10-02,kakao_pay,completed,2025-10,Thursday


In [23]:
monthly_sales = (
    order_sales
    .groupby('order_month', as_index=False)
    .agg(
        total_sales=('line_total', 'sum'),
        order_count=('order_id', 'nunique'),
    )
    .sort_values('order_month')
)

monthly_sales['avg_order_value'] = (
    monthly_sales['total_sales'] / monthly_sales['order_count']
).round(0)

monthly_sales


,order_month,total_sales,order_count,avg_order_value
0,2025-07,9561000.0,11,869182.0
1,2025-08,9362000.0,17,550706.0
2,2025-09,5299000.0,4,1324750.0
3,2025-10,12629000.0,17,742882.0
4,2025-11,9583000.0,16,598938.0
5,2025-12,8581000.0,12,715083.0
6,2026-01,8135000.0,9,903889.0
7,2026-02,7689000.0,12,640750.0
8,2026-03,7247000.0,13,557462.0
9,2026-04,11459000.0,14,818500.0


In [24]:
# Evidence 03 — completed 월별 금액·주문 수·객단가
print('Evidence 03: completed 월별 집계 (NaT/미래 포함)')
display(monthly_sales)
print('월별 합계:', f'{monthly_sales.total_sales.sum():,.0f}')

Evidence 03: completed 월별 집계 (NaT/미래 포함)


,order_month,total_sales,order_count,avg_order_value
0,2025-07,9561000.0,11,869182.0
1,2025-08,9362000.0,17,550706.0
2,2025-09,5299000.0,4,1324750.0
3,2025-10,12629000.0,17,742882.0
4,2025-11,9583000.0,16,598938.0
5,2025-12,8581000.0,12,715083.0
6,2026-01,8135000.0,9,903889.0
7,2026-02,7689000.0,12,640750.0
8,2026-03,7247000.0,13,557462.0
9,2026-04,11459000.0,14,818500.0


월별 합계: 141,340,000


### 실제 VS Code 화면 — Evidence 03

completed 월별 금액·주문 수·객단가. 실제 실행된 Notebook을 VS Code에서 열어 코드·실행 번호·출력을 캡처했다. 원본 JPEG를 그대로 저장했으며 합성·재그리기·이미지 편집을 하지 않았다.

![Evidence 03: completed 월별 금액·주문 수·객단가](images/step03_monthly.jpg)

### 결과 관찰
2026-08 금액은 16,980,000원, 주문 15건, 객단가 1,132,000원이다. 2026-07의 8,966,000원·8건·1,120,750원보다 금액은 약 89.38% 증가했다. 미래인 2027-01에 909,000원, NaT에 1,903,000원이 별도로 있다.

### 나의 해석과 판단
7월→8월 객단가 증가는 약 1.00%이고 주문 수는 87.5% 증가했다. 이 두 지표의 곱으로 금액 차이를 설명할 수 있지만 주문 수 증가의 원인은 알 수 없다. 미래·미상 그룹을 계절성 해석에 사용하지 않는다.

### 업무·분석적 의미
금액이 증가한 월에는 주문 유입과 주문 구성 중 어디를 후속 확인할지 구체화할 수 있다.

### 한계와 추가 확인 사항
미래 주문은 실제 관측 기간으로 해석하지 않는다. 1년 남짓의 짧은 표본이고 월별 기록 완전성도 검증되지 않아 계절성을 단정할 수 없다.

월별 매출이 특정 월에 높아졌다면 바로 원인을 단정하지 말고, 주문 수가 늘었는지 평균 주문 금액이 늘었는지, 특정 카테고리 매출이 늘었는지 추가로 확인해야 합니다.


## 14. 고객별 구매 금액 EDA

고객별 구매 금액은 우수 고객 분석의 출발점입니다. 하지만 한 번의 고액 구매 고객과 여러 번 반복 구매한 고객은 다르게 해석해야 합니다. 그래서 `total_sales`, `order_count`, `avg_order_value`를 함께 봅니다.


In [25]:
customer_sales_base = order_sales.merge(customers, on='customer_id', how='left', validate='many_to_one')
group_columns = ['customer_id', 'name', 'city']
customer_sales = (customer_sales_base.groupby(group_columns, as_index=False, dropna=False)
    .agg(order_count=('order_id', 'nunique'), total_sales=('line_total', 'sum'))
    .sort_values('total_sales', ascending=False))
customer_sales['avg_order_value'] = (customer_sales['total_sales'] / customer_sales['order_count']).round(0)
customer_sales.head(10)

,customer_id,name,city,order_count,total_sales,avg_order_value
1,3,이예우,서울,3,6702000.0,2234000.0
7,14,임서영,대구,4,5204000.0,1301000.0
57,90,윤서미,수원,3,4311000.0,1437000.0
59,93,윤도우,울산,4,4001000.0,1000250.0
47,65,오수민,광주,6,3977000.0,662833.0
66,104,윤혜원,서울,2,3966000.0,1983000.0
11,20,김성희,성남,4,3808000.0,952000.0
27,38,권우현,성남,4,3656000.0,914000.0
19,28,임민진,인천,4,3201000.0,800250.0
13,22,강영민,부산,3,3109000.0,1036333.0


In [26]:
# Evidence 04 — 합계 보존과 데이터 품질 민감도
print('Evidence 04: 동일 범위 총합 검증')
source_total = completed_items.line_total.sum()
checks = pd.DataFrame({'scope':['completed 원본','카테고리 (Unknown 포함)','월별 (NaT 포함)','고객별 (결측 그룹 포함)'],
    'amount':[source_total,category_sales.total_sales.sum(),monthly_sales.total_sales.sum(),customer_sales.total_sales.sum()]})
checks['difference'] = checks.amount-source_total
assert np.allclose(checks.amount, source_total)
display(checks)
known_only = sales_items.loc[sales_items.category.ne('Unknown'),'line_total'].sum()
valid_time = order_sales.loc[order_sales.order_date.notna() & order_sales.order_date.le(pd.Timestamp('2026-10-21'))]
print('카테고리 Unknown 제외 시 손실:', f'{source_total-known_only:,.0f}')
print('날짜 유효·기준일 이전 비교 합계:', f'{valid_time.line_total.sum():,.0f}')
print('날짜 미상 금액:', f'{order_sales.loc[order_sales.order_date.isna(),"line_total"].sum():,.0f}')
print('미래 주문 금액:', f'{order_sales.loc[order_sales.order_date.gt(pd.Timestamp("2026-10-21")),"line_total"].sum():,.0f}')
checks.to_csv(REPORT_DIR/'ch06_total_validation.csv',index=False)

Evidence 04: 동일 범위 총합 검증


,scope,amount,difference
0,completed 원본,141340000.0,0.0
1,카테고리 (Unknown 포함),141340000.0,0.0
2,월별 (NaT 포함),141340000.0,0.0
3,고객별 (결측 그룹 포함),141340000.0,0.0


카테고리 Unknown 제외 시 손실: 4,015,000
날짜 유효·기준일 이전 비교 합계: 138,528,000
날짜 미상 금액: 1,903,000
미래 주문 금액: 909,000


### 실제 VS Code 화면 — Evidence 04

동일 범위 총합과 날짜·미연결 민감도. 실제 실행된 Notebook을 VS Code에서 열어 코드·실행 번호·출력을 캡처했다. 원본 JPEG를 그대로 저장했으며 합성·재그리기·이미지 편집을 하지 않았다.

![Evidence 04: 동일 범위 총합과 날짜·미연결 민감도](images/step04_validation.jpg)

In [27]:
# completed 주문 커버리지: 상세가 전처리에서 전부 빠진 주문을 0원으로 대체하지 않는다.
completed_orders = orders.loc[orders.order_status.eq('completed')]
missing_detail_orders = completed_orders.loc[~completed_orders.order_id.isin(completed_items.order_id)]
orphan_items = order_items.loc[~order_items.order_id.isin(orders.order_id)]
print('completed 주문:', len(completed_orders), '금액 산출 가능한 주문:', order_sales.order_id.nunique())
print('상세가 없는 completed 주문:', missing_detail_orders.order_id.tolist())
print('주문 미연결 상세:',len(orphan_items),'행 / 금액:',orphan_items.line_total.sum())
print('고객 미연결 completed 금액:',customer_sales_base.loc[customer_sales_base.name.isna(),'line_total'].sum())
assert order_sales.order_id.nunique() == 179
assert len(missing_detail_orders) == 1


completed 주문: 180 금액 산출 가능한 주문: 179
상세가 없는 completed 주문: [175]
주문 미연결 상세: 2 행 / 금액: 922000.0
고객 미연결 completed 금액: 376000.0


### 핵심 집계 총합과 범위 검증

- completed 원본 합계: **141,340,000원** (446개 상세, 상세가 남은 주문 179건)
- 카테고리별 합계: **141,340,000원**, Unknown 포함
- 월별 합계: **141,340,000원**, NaT·미래 포함
- 고객별 합계: **141,340,000원**, 고객 미연결도 결측 그룹으로 보존
- 총합 일치 여부: **일치**, 모든 차이 0원
- 날짜 유효·2026-10-21 이전 비교 범위: **138,528,000원**
- 원본과 비교 범위의 차이: 날짜 미상 1,903,000원 + 미래 909,000원 = **2,812,000원**

### 결과 관찰
완료 주문 180건 중 order_id 175는 전처리 후 상세가 없어 이번 금액 분석의 분모는 179건이다. 주문 미연결 상세 2행(922,000원)은 상태를 확인할 수 없어 completed 범위에 넣지 않는다. 고객 미연결 completed 금액 376,000원은 보존한다.

### 나의 해석과 판단
합계 일치는 같은 범위 안에서 금액이 보존되었다는 증거다. 주문 175의 실제 금액이나 미연결 상세 상태까지 정상이라는 뜻은 아니다. 상세가 없는 주문은 실제 0원이라고 가정하지 않는다. 날짜 결측 그룹이 누락되는 문제는 `.fillna('NaT')`로 명시적으로 해결했다.

### 업무·분석적 의미
누락을 숨긴 합계로 지표를 비교하는 것을 방지하고 원천 데이터 확인 대상과 금액 영향도를 제시할 수 있다.

### 한계와 추가 확인 사항
분석 기간을 좁히면 원본·카테고리·월별을 모두 같은 필터로 재계산해야 한다. 고객 미연결 금액은 지역별 고객당 지표의 분모로 섞지 않는다.

## 15. EDA 결과를 다음 질문으로 연결하기

EDA의 가치는 표를 만드는 데서 끝나지 않습니다. 각 결과가 다음 질문으로 이어져야 합니다.


In [28]:
eda_result_summary = pd.DataFrame({
    'question': [
        '고객은 어느 도시에 많이 분포하는가?',
        '어떤 카테고리의 상품이 많은가?',
        '카테고리별 매출은 어떻게 다른가?',
        '월별 매출과 주문 수는 어떻게 변하는가?',
        '구매 금액이 높은 고객은 누구인가?',
    ],
    'result_table': [
        'customer_city',
        'product_category',
        'category_sales',
        'monthly_sales',
        'customer_sales',
    ],
    'next_question': [
        '도시별 구매 금액도 차이가 있는가?',
        '상품 수가 많은 카테고리가 매출도 높은가?',
        '매출 차이가 수량 때문인가 단가 때문인가?',
        '특정 월의 매출 변화는 어떤 카테고리 때문인가?',
        '고액 구매 고객은 반복 구매 고객인가?',
    ],
})

eda_result_summary


,question,result_table,next_question
0,고객은 어느 도시에 많이 분포하는가?,customer_city,도시별 구매 금액도 차이가 있는가?
1,어떤 카테고리의 상품이 많은가?,product_category,상품 수가 많은 카테고리가 매출도 높은가?
2,카테고리별 매출은 어떻게 다른가?,category_sales,매출 차이가 수량 때문인가 단가 때문인가?
3,월별 매출과 주문 수는 어떻게 변하는가?,monthly_sales,특정 월의 매출 변화는 어떤 카테고리 때문인가?
4,구매 금액이 높은 고객은 누구인가?,customer_sales,고액 구매 고객은 반복 구매 고객인가?


## 16. EDA 결과 저장하기

EDA 결과표를 CSV로 저장하면 다음 장의 시각화나 보고서 작성에서 다시 사용할 수 있습니다.


In [29]:
customer_city.to_csv(REPORT_DIR / 'ch06_customer_city.csv', index=False, encoding='utf-8-sig')
product_category.to_csv(REPORT_DIR / 'ch06_product_category.csv', index=False, encoding='utf-8-sig')
category_sales.to_csv(REPORT_DIR / 'ch06_category_sales.csv', index=False, encoding='utf-8-sig')
monthly_sales.to_csv(REPORT_DIR / 'ch06_monthly_sales.csv', index=False, encoding='utf-8-sig')
customer_sales.to_csv(REPORT_DIR / 'ch06_customer_sales.csv', index=False, encoding='utf-8-sig')
eda_result_summary.to_csv(REPORT_DIR / 'ch06_eda_questions.csv', index=False, encoding='utf-8-sig')

list(REPORT_DIR.glob('ch06_*.csv'))


[PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_segment_summary.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_monthly_sales.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_monthly_category.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_product_category.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_eda_questions.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_city_comparison.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_total_validation.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_category_sales.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_payment_aov.csv'),
 PosixPath('/Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_quantity_

## 17. EDA 요약 보고서 만들기

간단한 Markdown 요약 보고서를 만들어 봅니다. 보고서에는 분석 목적, 주요 질문, 핵심 결과, 추가 질문, 해석 시 주의사항을 담습니다.


In [30]:
summary_text = f'''# Chapter 6 EDA 요약 보고서

## 1. 분석 목적

전처리된 온라인 쇼핑몰 데이터를 사용해 고객, 상품, 주문, 매출 관점의 기본 현황을 탐색했습니다.

## 2. 주요 분석 질문

```text
{questions.to_string(index=False)}
```

## 3. 카테고리별 매출 요약

```text
{category_sales.head(10).to_string(index=False)}
```

## 4. 월별 매출 요약

```text
{monthly_sales.to_string(index=False)}
```

## 5. 고객별 구매 금액 상위 10명

```text
{customer_sales.head(10).to_string(index=False)}
```

## 6. 추가 분석 질문

```text
{eda_result_summary.to_string(index=False)}
```

## 7. 해석 시 주의사항

- EDA 결과는 최종 결론이 아니라 추가 분석을 위한 관찰 결과입니다.
- 매출이 높은 카테고리가 반드시 선호도가 높은 카테고리라는 뜻은 아닙니다.
- 월별 매출 변화의 원인을 설명하려면 프로모션, 계절성, 신규 상품 등의 추가 정보가 필요합니다.
- 고객별 구매 금액은 주문 횟수와 평균 주문 금액을 함께 해석해야 합니다.
'''

report_path = REPORT_DIR / 'ch06_eda_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('EDA 요약 보고서 저장 완료:', report_path)


EDA 요약 보고서 저장 완료: /Users/kjj/MyLLMGit/llm-data-analysis-study/chapter06/reports/ch06_eda_summary.md


## 18. 소스 모듈로 같은 EDA 실행하기

위에서 노트북으로 한 단계씩 실행한 EDA는 `src/eda.py`에 함수로 정리되어 있습니다. 반복 실행하거나 프로젝트 코드로 관리하려면 노트북보다 소스 모듈을 사용하는 것이 좋습니다.


In [31]:
from src.eda import (
    build_eda_report,
    load_processed_sales_data,
    run_basic_eda,
    save_eda_outputs,
)

module_data = load_processed_sales_data(PROCESSED_DIR)
module_results = run_basic_eda(module_data)

module_results.keys()


dict_keys(['questions', 'customer_city', 'customer_gender', 'product_category', 'category_price', 'order_status', 'payment_method', 'category_sales', 'monthly_sales', 'customer_sales', 'eda_result_summary'])

In [32]:
module_results['category_sales'].head()


,category,total_quantity,total_sales,sales_ratio
7,패션,290.0,32940000.0,23.31
5,식품,226.0,25035000.0,17.71
6,전자기기,181.0,21590000.0,15.28
2,뷰티,180.0,17751000.0,12.56
1,도서,174.0,17663000.0,12.50


In [33]:
# 독립 모듈의 집계와 Notebook 결과를 전체 표 단위로 비교한다.
from pandas.testing import assert_frame_equal
for key in ['category_sales','monthly_sales','customer_sales']:
    cols = {'category_sales':['category'],'monthly_sales':['order_month'],'customer_sales':['customer_id']}[key]
    a = globals()[key].sort_values(cols).reset_index(drop=True)
    b = module_results[key].sort_values(cols).reset_index(drop=True)
    assert_frame_equal(a,b,check_dtype=False)
print('모듈 ↔ Notebook 카테고리/월별/고객별 전체 표 일치')

모듈 ↔ Notebook 카테고리/월별/고객별 전체 표 일치


## 19. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 EDA 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/run_eda.py
```

이 스크립트는 6장 EDA 결과 CSV와 `reports/ch06_eda_summary.md`를 자동으로 저장합니다.


## 20. LLM과 함께 질문을 확장하기

LLM은 EDA 질문을 확장하고 결과 해석 문장을 다듬는 데 도움이 됩니다. 하지만 LLM이 제안한 질문이 실제 데이터로 답할 수 있는지는 반드시 사람이 확인해야 합니다.

```text
온라인 쇼핑몰 데이터로 EDA를 수행하려고 합니다.

데이터셋:
- customers: customer_id, gender, age, city, signup_date
- products: product_id, product_name, category, price
- orders: order_id, customer_id, order_date, payment_method, order_status
- order_items: order_id, product_id, quantity, unit_price, line_total

요청:
1. 현재 데이터로 검증 가능한 EDA 질문 10개를 만들어 주세요.
2. 각 질문에 필요한 데이터셋과 pandas 기능을 함께 정리해 주세요.
3. 데이터에 없는 내용은 추측하지 마세요.
4. 각 질문을 계산 가능한 지표로 바꿔 주세요.
```


## 21. LLM 결과 해석 요청 예시

EDA 결과 해석을 요청할 때는 원인 단정을 막는 조건을 넣는 것이 좋습니다.

```text
다음은 온라인 쇼핑몰 카테고리별 매출 요약 결과입니다.

category,total_quantity,total_sales,sales_ratio
전자기기,320,12500000,42.5
생활용품,510,7800000,26.5
패션,260,6200000,21.1
식품,430,2900000,9.9

이 결과를 보고서에 넣을 수 있도록 해석해 주세요.

조건:
- 데이터에 없는 원인을 단정하지 말 것
- 원인 설명은 가설로 표현할 것
- 추가로 확인해야 할 분석 질문을 제안할 것
- 관찰, 가설, 추가 분석을 구분할 것
```


## 22. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
3. 결제수단별 평균 주문 금액을 계산하세요.
4. 월별·카테고리별 매출 요약표를 만들어 보세요.
5. 고객별 `order_count`와 `avg_order_value`를 함께 보고 반복 구매 고객과 고액 단발 구매 고객을 구분하는 기준을 생각해 보세요.
6. LLM에게 현재 데이터로 답할 수 없는 질문을 답할 수 있는 질문으로 바꾸게 하는 프롬프트를 작성해 보세요.


In [34]:
# 과제 1. 도시별 고객 수와 도시별 총 구매 금액을 비교하세요.
city_spend = customer_sales_base.assign(city=lambda x: x.city.fillna('UnmatchedCustomer')).groupby('city', as_index=False).agg(
    total_sales=('line_total','sum'), buyers=('customer_id','nunique'))
city_comparison = customer_city.merge(city_spend,on='city',how='outer').fillna(0)
city_comparison['sales_per_registered_customer'] = city_comparison.total_sales / city_comparison.customer_count.replace(0,np.nan)
city_comparison['sales_per_buyer'] = city_comparison.total_sales / city_comparison.buyers.replace(0,np.nan)
assert np.isclose(city_comparison.total_sales.sum(), completed_items.line_total.sum())
city_comparison.sort_values('total_sales',ascending=False)

,city,customer_count,customer_ratio,total_sales,buyers,sales_per_registered_customer,sales_per_buyer
6,부산,23.0,15.33,24916000.0,16,1.083304e+06,1.557250e+06
7,서울,18.0,12.00,22211000.0,11,1.233944e+06,2.019182e+06
3,광주,21.0,14.00,19866000.0,16,9.460000e+05,1.241625e+06
11,인천,16.0,10.67,15723000.0,12,9.826875e+05,1.310250e+06
4,대구,21.0,14.00,14781000.0,12,7.038571e+05,1.231750e+06
9,수원,10.0,6.67,12220000.0,8,1.222000e+06,1.527500e+06
10,울산,17.0,11.33,11545000.0,9,6.791176e+05,1.282778e+06
8,성남,9.0,6.00,11288000.0,5,1.254222e+06,2.257600e+06
2,고양,8.0,5.33,7222000.0,6,9.027500e+05,1.203667e+06
0,Unknown,2.0,1.33,820000.0,1,4.100000e+05,8.200000e+05


In [35]:
# 과제 2. 상품 수가 많은 카테고리가 매출도 높은지 확인하세요.
category_comparison = product_category.merge(category_sales,on='category',how='outer')
category_comparison['weighted_unit_price'] = category_comparison.total_sales / category_comparison.total_quantity
known = category_comparison.dropna(subset=['product_count'])
print('카테고리 상품 수·금액 Spearman 상관:', known.product_count.corr(known.total_sales,method='spearman'))
category_comparison.sort_values('total_sales',ascending=False)

카테고리 상품 수·금액 Spearman 상관: 0.7637626158259735


,category,product_count,product_ratio,total_quantity,total_sales,sales_ratio,weighted_unit_price
7,패션,16.0,16.67,290.0,32940000.0,23.31,113586.206897
5,식품,16.0,16.67,226.0,25035000.0,17.71,110774.336283
6,전자기기,14.0,14.58,181.0,21590000.0,15.28,119281.767956
2,뷰티,17.0,17.71,180.0,17751000.0,12.56,98616.666667
1,도서,13.0,13.54,174.0,17663000.0,12.50,101511.494253
4,스포츠,10.0,10.42,165.0,13237000.0,9.37,80224.242424
3,생활용품,10.0,10.42,125.0,9109000.0,6.44,72872.000000
0,Unknown,NaN,NaN,34.0,4015000.0,2.84,118088.235294


In [36]:
# 과제 3. 결제수단별 평균 주문 금액을 계산하세요.
# 상세 행의 평균이 아니라 먼저 주문별로 합산한다.
order_amounts = order_sales.groupby(['order_id','payment_method'],as_index=False).agg(order_total=('line_total','sum'))
payment_aov = order_amounts.groupby('payment_method',as_index=False).agg(
    order_count=('order_id','nunique'),total_sales=('order_total','sum'),avg_order_value=('order_total','mean'))
assert np.isclose(payment_aov.total_sales.sum(), completed_items.line_total.sum())
payment_aov.sort_values('avg_order_value',ascending=False)

,payment_method,order_count,total_sales,avg_order_value
1,card,40,37542000.0,938550.000000
0,bank_transfer,49,39461000.0,805326.530612
2,kakao_pay,50,36861000.0,737220.000000
3,naver_pay,40,27476000.0,686900.000000


In [37]:
# 과제 4. 월별·카테고리별 매출 요약표를 만들어 보세요.
monthly_category_base = order_sales.merge(products[['product_id','category']],on='product_id',how='left',validate='many_to_one')
monthly_category_base['category'] = monthly_category_base.category.fillna('Unknown')
monthly_category = monthly_category_base.pivot_table(index='order_month',columns='category',values='line_total',aggfunc='sum',fill_value=0)
assert np.isclose(monthly_category.to_numpy().sum(), completed_items.line_total.sum())
monthly_category

category,Unknown,도서,뷰티,생활용품,스포츠,식품,전자기기,패션
order_month,,,,,,,,
2025-07,82000.0,1040000.0,1133000.0,406000.0,295000.0,2123000.0,1319000.0,3163000.0
2025-08,470000.0,1144000.0,1050000.0,983000.0,417000.0,505000.0,2547000.0,2246000.0
2025-09,780000.0,380000.0,125000.0,735000.0,438000.0,528000.0,1157000.0,1156000.0
2025-10,82000.0,1570000.0,661000.0,1334000.0,1426000.0,3088000.0,1781000.0,2687000.0
2025-11,420000.0,583000.0,1581000.0,528000.0,1271000.0,2221000.0,1035000.0,1944000.0
2025-12,468000.0,34000.0,884000.0,489000.0,791000.0,2048000.0,1408000.0,2459000.0
2026-01,0.0,74000.0,376000.0,804000.0,2359000.0,2465000.0,590000.0,1467000.0
2026-02,0.0,1827000.0,275000.0,646000.0,1009000.0,1193000.0,1562000.0,1177000.0
2026-03,140000.0,1075000.0,728000.0,233000.0,364000.0,2792000.0,1378000.0,537000.0


In [38]:
# 과제 5 — 표본 내 기준으로 반복 구매와 고액 단발 구매를 구분한다.
linked_customer = customer_sales.customer_id.isin(customers.customer_id)
threshold = customer_sales.loc[linked_customer,'avg_order_value'].quantile(.75)
customer_segments = customer_sales.assign(segment=np.select(
    [~linked_customer, customer_sales.order_count.ge(2), customer_sales.order_count.eq(1) & customer_sales.avg_order_value.ge(threshold)],
    ['고객 미연결','반복 구매 (2회 이상)','고액 단발 (1회·객단가 상위25%)'],default='기타 단발'))
segment_summary = customer_segments.groupby('segment',as_index=False).agg(customers=('customer_id','nunique'),total_sales=('total_sales','sum'))
print('고액 단발 기준 (구매자별 객단가 75% 분위):', threshold)
display(segment_summary)
# 수량 극단값은 삭제 대신 동일 범위에서 제외한 민감도만 비교한다.
quantity_sensitivity = sales_items.loc[sales_items.quantity.le(20)].groupby('category',as_index=False).agg(total_sales=('line_total','sum')).sort_values('total_sales',ascending=False)
print('수량 20 초과 제외 시 카테고리 순위:')
display(quantity_sensitivity)
for name in ['city_comparison','category_comparison','payment_aov','monthly_category','segment_summary','quantity_sensitivity']:
    globals()[name].to_csv(REPORT_DIR/f'ch06_{name}.csv',index=name=='monthly_category',encoding='utf-8-sig')

고액 단발 기준 (구매자별 객단가 75% 분위): 1026500.0


,segment,customers,total_sales
0,고객 미연결,1,376000.0
1,고액 단발 (1회·객단가 상위25%),11,14770000.0
2,기타 단발,34,17454000.0
3,반복 구매 (2회 이상),53,108740000.0


수량 20 초과 제외 시 카테고리 순위:


,category,total_sales
7,패션,28140000.0
5,식품,25035000.0
6,전자기기,21590000.0
2,뷰티,17751000.0
1,도서,17663000.0
4,스포츠,13237000.0
3,생활용품,9109000.0
0,Unknown,4015000.0


In [39]:
# Evidence 05 — 확장 질문을 실제 데이터로 검증
print('Evidence 05: 후속 질문 검증')
print('7월→8월 카테고리별 금액 증감:')
category_delta = (monthly_category.loc['2026-08'] - monthly_category.loc['2026-07']).sort_values(ascending=False)
display(category_delta.rename('sales_change').to_frame())
assert np.isclose(category_delta.sum(),16_980_000-8_966_000)
display(payment_aov.round(2))
display(segment_summary)
print('고객 미연결을 별도 그룹으로 분리 / 결제수단별 금액 총합 검증 통과')

Evidence 05: 후속 질문 검증
7월→8월 카테고리별 금액 증감:


,sales_change
category,
전자기기,3212000.0
패션,2307000.0
도서,1543000.0
스포츠,926000.0
생활용품,917000.0
Unknown,312000.0
뷰티,-587000.0
식품,-616000.0


,payment_method,order_count,total_sales,avg_order_value
0,bank_transfer,49,39461000.0,805326.53
1,card,40,37542000.0,938550.00
2,kakao_pay,50,36861000.0,737220.00
3,naver_pay,40,27476000.0,686900.00


,segment,customers,total_sales
0,고객 미연결,1,376000.0
1,고액 단발 (1회·객단가 상위25%),11,14770000.0
2,기타 단발,34,17454000.0
3,반복 구매 (2회 이상),53,108740000.0


고객 미연결을 별도 그룹으로 분리 / 결제수단별 금액 총합 검증 통과


### 실제 VS Code 화면 — Evidence 05

카테고리별 증감과 결제수단 객단가. 실제 실행된 Notebook을 VS Code에서 열어 코드·실행 번호·출력을 캡처했다. 원본 JPEG를 그대로 저장했으며 합성·재그리기·이미지 편집을 하지 않았다.

![Evidence 05: 카테고리별 증감과 결제수단 객단가](images/step05_followup.jpg)

## 답안 3. 관찰 → 가설 → 추가 검증

### 사례 1. 패션 금액 1위
- **관찰:** 패션 금액 32,940,000원·수량 290개로 1위다. 전자기기는 가중 거래단가가 더 높다.
- **가설:** 패션은 단가보다 판매 수량과 고수량 주문의 영향이 클 수 있다.
- **추가 검증:** 거래 단가를 금액/수량으로 계산하고 수량 20 초과를 제외해 비교했다. 패션 금액은 28,140,000원으로 4,800,000원 줄지만 1위는 유지된다.
- **판단:** 특정 고수량 상세 하나가 금액을 높였지만 1위가 그 행에만 의존하지는 않는다. 선호도·반복 수요의 증거로는 부족하다. 다음에는 상품별·주문별 집중도를 본다.

### 사례 2. 2026년 8월 증가
- **관찰:** 7월 대비 금액 8,014,000원 증가. 주문은 8→15건, 객단가는 1,120,750→1,132,000원이다.
- **가설:** 구매 빈도·주문 유입 증가 및 거래 품목 구성 변화가 함께 나타났을 수 있다.
- **추가 검증:** 월×카테고리 표를 비교했다. 전자기기 +3,212,000원, 패션 +2,307,000원, 도서 +1,543,000원이며 뷰티·식품은 감소했다. 증감 합은 전체 차이와 일치한다.
- **판단:** 증가가 전 카테고리에 동일하게 나타난 것은 아니다. 7월 전자기기 금액 0원이 무수요인지 누락인지 확인해야 한다. 광고·계절성의 영향은 알 수 없다.

### 사례 3. 상품 수와 금액 순위
- **관찰:** 뷰티가 상품 17개로 가장 많지만 금액은 17,751,000원으로 알려진 카테고리 중 4위다. 패션·식품은 각 16개다. 알려진 7개 카테고리의 Spearman 상관은 약 0.764다.
- **가설:** 상품 구색이 많은 카테고리에 판매 기회가 많을 수 있지만 판매 수량·거래 단가·인기 상품 집중도가 순위를 달리했을 수 있다.
- **추가 검증:** 상품 수와 수량·가중 거래단가를 함께 표로 비교했다. 다음에는 미판매 상품 비율과 상품별 거래 분포를 계산한다.
- **판단:** 소수 그룹의 탐색적 상관이다. 상품 수를 늘리면 금액이 증가한다고 결론 내리지 않는다.

### 과제 1·3·5 해석
부산은 등록 고객 23명·금액 24,916,000원으로 모두 1위지만, 고객당 금액은 서울·성남 등과 순서가 다르다. 등록 고객 수는 현재 스냅샷이므로 구매 전환율의 확정 분모가 아니다. `Unknown`은 도시가 없는 등록 고객, `UnmatchedCustomer`는 고객 키 미연결로 구분했다. 후자는 등록 고객당 금액을 계산하지 않는다.

결제수단별 객단가는 card 938,550원, bank_transfer 약 805,326.53원, kakao_pay 737,220원, naver_pay 686,900원이다. 상세 행 평균 대신 주문별 금액을 먼저 계산했다. 결제수단의 효과가 아니라 표본 내 차이이며 상품 구성·금액대·사용자 특성 확인이 필요하다.

반복 구매는 이 표본에서 완료 주문 2회 이상으로 정했다. 고액 단발은 1회 구매이면서 연결 가능한 구매자들의 객단가 75% 분위 이상이다. 고객 미연결은 별도 그룹으로 분리했다. 관측기간 밖 재구매와 다른 채널은 모르므로 영구적인 VIP·충성 고객 분류로 사용하지 않는다.

### 업무·분석적 의미 및 한계
재고·프로모션·고객 관리 검토의 후보를 좁힐 수 있다. 하지만 이 연습 데이터에서 수익·만족도·마케팅 성과를 추정하지 않는다. 금액은 원가·배송비·수수료·할인 데이터가 없어 이익이 아니다.

## 23. 정리

이번 장에서는 다음 내용을 실습했습니다.

- EDA에서 관찰, 가설, 결론 구분하기
- 현재 데이터로 답할 수 있는 질문 만들기
- 고객, 상품, 주문 데이터의 기본 분포 확인
- 주문 상세, 상품, 주문, 고객 데이터를 병합해 매출 분석하기
- 카테고리별 매출, 월별 매출, 고객별 구매 금액 계산하기
- EDA 결과를 다음 질문으로 연결하기
- EDA 결과 CSV와 Markdown 보고서 저장하기
- `src/eda.py`와 `scripts/run_eda.py`로 반복 실행 가능한 소스 구조 만들기

다음 장에서는 EDA에서 만든 질문과 집계표를 바탕으로 데이터 시각화를 다룹니다.


## 답안 4. LLM 질문 확장

**Safe Context:** 가상 쇼핑몰의 컬럼명과 집계만 사용했다. customers의 키·나이·도시, products의 키·카테고리·가격, orders의 키·날짜·상태·결제수단, order_items의 키·수량·거래단가·line_total을 설명했다. 집계는 completed 446상세/141,340,000원, Unknown 4,015,000원, 7월 8,966,000원/8건, 8월 16,980,000원/15건이다. 고객 이름이나 개별 행을 질문에 포함하지 않았다.

**사용 도구·기록:** 2026-10-21 이 대화의 Codex가 아래 조건으로 작성한 질문 후보를 기록했다. 별도 Gemini API를 실행한 것처럼 표시하지 않으며 재실행에 API 키가 필요 없다. 아래 판단·수정도 Codex가 작성한 분석 초안이다.

**Prompt:**
```text
위 스키마와 집계만 사용해 다음 EDA 질문 5개를 제안하라.
각 질문의 필요 데이터, 계산 가능한 지표와 검증 방법을 써라.
광고 효과와 고객 이탈처럼 현재 답할 수 없는 질문은 필요한 외부 정보와
현재 데이터로 답할 수 있는 대체 질문을 구분하라.
원인을 단정하지 말고 completed 필터, 분모와 결측 처리 기준을 명시하라.
```

**LLM 제안 요약 및 검토:**

| LLM 제안 | 실제 가능 여부 | 사용 여부 | 판단 이유·수정 내용 |
|---|---|---|---|
| 패션 금액 1위는 고수량 주문 때문인가? | 수량·금액이 있어 민감도 비교 가능 | 수정 후 사용 | 원인 단정 대신 수량 20 초과 포함/제외 비교. 1위 유지 확인 |
| 8월 증가에 어떤 카테고리가 기여했는가? | 월·상품 키·금액 있음 | 사용 | 월×카테고리 증감 합과 전체 차이 일치 검증 |
| 결제수단마다 객단가가 다른가? | 결제수단·주문 ID·금액 있음 | 사용 | 상세 평균을 피하고 주문 단위 합산 후 평균 |
| 광고가 8월 금액을 늘렸는가? | 광고 노출·집행·대조군 없음 | 보류 | 대체: 주문 수·객단가·카테고리 구성 변화 확인. 광고 효과는 별도 실험 필요 |
| 구매하지 않은 고객은 이탈했는가? | 관측 종료·다른 채널·활동 정보 부족 | 수정 후 사용 | 이탈 판정 대신 관측 구간 내 반복 구매/고액 단발 탐색. 고객 미연결 별도 |

### 과제 6. 답할 수 없는 질문을 바꾸는 Prompt
```text
“광고 때문에 8월 매출이 올랐나?”와 “어떤 고객이 이탈했나?”를 검토하라.
현재 테이블에 광고·세션·다른 채널·완전한 관측 종료 정보는 없다.
먼저 식별 불가능한 원인과 필요한 추가 데이터를 적어라.
그다음 completed 주문의 월별 주문 수·객단가·카테고리 증감,
관측 구간 내 구매 횟수처럼 계산 가능한 질문으로 바꾸어라.
분석 범위, 분모, 누락·미래 날짜 처리와 총합 검증을 명시하라.
```

### 결과 관찰
5개 후보 중 2개 사용, 2개 수정 후 사용, 광고 인과 질문 1개 보류로 정리했다. 채택 질문은 앞의 실제 집계로 확인했다.

### 나의 해석과 판단
LLM의 원인 표현과 실제 계산 가능성을 분리했다. 집계상 차이가 존재한다는 것과 그 차이의 원인을 알 수 있다는 것은 다르다.

### 업무·분석적 의미
데이터가 없는 질문에 시간을 쓰지 않고, 원천 확인·후속 계산·추가 수집이 필요한 질문으로 나눌 수 있다.

### 한계와 추가 확인 사항
AI 제안·답안 초안이므로 제출자는 해석을 읽고 자신의 판단과 맞는지 검토해야 한다. 고객 이탈·마케팅 효과·수익성은 현재 표만으로 확정할 수 없다.

In [40]:
# Evidence 06 — LLM 제안 채택·수정·보류와 실행 검증
print('Evidence 06: LLM 질문 검토 및 재현성')
llm_review = pd.DataFrame({
    '질문':['고수량 민감도','8월 카테고리 증감','결제수단 객단가','광고 인과 효과','고객 이탈'],
    '판단':['수정 후 사용','사용','사용','보류','수정 후 사용'],
    '검증':['제외 후 패션 1위 유지','증감 합계 8,014,000원','주문별 합산 후 평균','광고 데이터 없음','관측 구간 구매 횟수로 수정']})
display(llm_review)
assert np.isclose(category_delta.sum(),8_014_000)
assert np.isclose(payment_aov.total_sales.sum(),source_total)
assert np.isclose(segment_summary.total_sales.sum(),source_total)
print('세 검증 통과 / API 키 없이 재실행 가능')
llm_review.to_csv(REPORT_DIR/'ch06_llm_review.csv',index=False,encoding='utf-8-sig')

Evidence 06: LLM 질문 검토 및 재현성


,질문,판단,검증
0,고수량 민감도,수정 후 사용,제외 후 패션 1위 유지
1,8월 카테고리 증감,사용,"증감 합계 8,014,000원"
2,결제수단 객단가,사용,주문별 합산 후 평균
3,광고 인과 효과,보류,광고 데이터 없음
4,고객 이탈,수정 후 사용,관측 구간 구매 횟수로 수정


세 검증 통과 / API 키 없이 재실행 가능


### 실제 VS Code 화면 — Evidence 06

LLM 제안 사용·수정·보류와 실행 검증. 실제 실행된 Notebook을 VS Code에서 열어 코드·실행 번호·출력을 캡처했다. 원본 JPEG를 그대로 저장했으며 합성·재그리기·이미지 편집을 하지 않았다.

![Evidence 06: LLM 제안 사용·수정·보류와 실행 검증](images/step06_llm_review.jpg)

## 답안 5. 다음 분석 우선순위

1. **원천 품질 확인:** 상품 미연결 4,015,000원, 날짜 미상 1,903,000원, 미래 금액 909,000원, 고객 미연결 376,000원과 상세 없는 completed 1건의 사유를 확인한다. 서로 겹칠 수 있는 품질 금액이므로 단순 합산해 손실액으로 부르지 않는다.
2. **월 변화 분해:** 날짜 유효·기준일 이전 범위에서 7월 전자기기 0원과 8월 증가를 주문·상품 단위로 확인한다. 자료 누락 여부를 먼저 보고 신규/반복 구매자 구성을 비교한다.
3. **카테고리 집중도와 고객 구매 패턴:** 패션의 상품별 기여도, 고수량 주문 집중도를 확인하고 반복 구매자와 고액 단발을 나누어 객단가·주문 수를 본다.

### 가장 먼저 확인하고 싶은 이유
연결·날짜 문제는 분석 범위와 분모 자체를 바꾸기 때문이다. 품질 확인 전에 특정 카테고리 투자나 고객 캠페인 결정을 내리면 지표가 바뀔 수 있다.

### 현재 데이터만으로 단정할 수 없는 것
광고·프로모션의 인과 효과, 계절성, 고객 만족도·이탈·충성도, 수익성, 데이터 누락이 없는 실제 전체 매출은 확정할 수 없다. 미래 날짜가 예약 주문인지 입력 오류인지도 원천 담당자의 확인이 필요하다.

### 업무적으로 확인할 가치가 있는 질문
“미연결 상품을 복구해도 카테고리 순위가 유지되는가?”, “7월 전자기기 0원은 실제 무거래인가?”, “상세 없는 completed 주문은 전처리 제외의 결과인가?”를 담당자에게 먼저 확인할 가치가 있다.

## 최종 체크

- [x] 공식 Notebook의 전체 코드 실행 결과를 저장했다.
- [x] 질문 5개에 범위·데이터·지표·검증을 연결했다.
- [x] 모든 금액 집계에 completed 범위를 적용했다.
- [x] 원본·카테고리·월별·고객별 총합을 검증했다.
- [x] 관찰·가설·추가 검증·판단과 한계를 구분했다.
- [x] 공식 과제 1~6 및 LLM 제안 사용·수정·보류를 작성했다.
- [x] 독립 모듈과 Notebook의 집계 전체 표를 대조했다.

**재현 명령:** 저장소 루트에서 `.venv/bin/python chapter06/scripts/run_eda.py`를 실행하면 같은 completed 범위의 CSV를 `chapter06/reports/`에 저장한다. Notebook 커널을 재시작한 뒤 위에서 아래로 실행해도 동일 결과가 나와야 한다. 실제 VS Code 화면 증거는 아래 첨부한다.